# Clinical RAG and precision effects

**Goal:** Reproduce paired clinical estimates with cases shared across systems.

Run cells from top to bottom. Default cells work offline; model execution is an explicit opt-in and writes only to ignored `outputs/`.

## 1. Set up paths

Find the repository and load the analysis helpers.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/final").is_dir(), "Run from the repository or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from sleepinn_study.io import read_json, read_jsonl, output_directory
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


## 2. Estimate model-specific paired RAG effects

Bootstrap within source books; use exact McNemar tests for binary paired scores. P values are exploratory and unadjusted. Missing scores remain excluded with explicit denominators and bounds.

In [ ]:
from sleepinn_study.statistics import clinical_rag_table
comparison = pd.read_csv(ROOT / "results/analysis/clinical/judge_comparison_19_systems.csv")
effects = clinical_rag_table(comparison)
display(effects)
published = pd.read_csv(ROOT / "results/analysis/clinical/paired_rag_effects.csv")
check = effects.merge(published, left_on=["model_id", "precision"], right_on=["model", "precision"], suffixes=("_new", "_saved"))
assert len(check) == 19
assert (check.delta_pp_new - check.delta_pp_saved).abs().max() < 1e-9

## 3. Plot RAG and no-RAG scores

Both bars for each system use the same retained cases. Sort systems by their highest observed score; do not give closed models a separate scale.

In [ ]:
plot = effects.set_index(effects.model_id.str.split("/").str[-1] + " · " + effects.precision)
plot["best"] = plot[["no_rag_pct", "with_rag_pct"]].max(axis=1)
plot = plot.sort_values("best")
ax = plot[["no_rag_pct", "with_rag_pct"]].plot.barh(figsize=(10, 9), color=["#a3b8c2", "#327c9d"])
ax.set(xlabel="Clinical reference agreement on consensus-retained pairs (%)", xlim=(0,100), ylabel="")
plt.tight_layout()
plt.savefig(output_directory("figures") / "clinical_rag_no_rag.png")
plt.show()

## 4. Reproduce aggregate RAG and precision contrasts

Use 30,000 case-cluster bootstrap samples within the six books and 100,000 shared-case sign permutations. Each resample keeps the same case together across all systems. Equal weight per system is an explicit estimand, not a meta-analysis of independent studies.

In [ ]:
from sleepinn_study.aggregate import aggregate_clinical
aggregate = aggregate_clinical(comparison)
display(pd.DataFrame(aggregate))
saved = read_json(ROOT / "results/analysis/clinical/AGGREGATE_RESULTS.json")["results"]
for current, original in zip(aggregate, saved):
    assert abs(current["delta_pp"] - original["delta_pp"]) < 1e-10
    assert abs(current["p_unadjusted"] - original["case_cluster_sign_permutation_p_unadjusted"]) < 1e-10